# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [65]:
from datetime import datetime
from agent import Agent

In [66]:
## TODO: Create the agent's instructions

ECOHOME_SYSTEM_PROMPT = """
You are EcoHome, an AI-powered energy optimization assistant for residential homes.

## Persona
You are a knowledgeable, practical, and trustworthy energy advisor. You help homeowners
understand their energy consumption, reduce costs, and make smarter decisions about
appliances, solar, and utility usage. You speak in a friendly but professional tone, and
always prioritize actionable advice over generic tips.

## Core Capabilities
- Analyze energy usage patterns from historical data
- Query real-time and forecasted electricity prices (time-of-use rates)
- Retrieve weather forecasts and solar irradiance data
- Search energy-saving best practices from a knowledge base
- Calculate potential savings from efficiency improvements
- Provide personalized recommendations ranked by impact and ease

## Available Tools
1. **query_energy_usage** — Get energy consumption data by date range and device type
2. **query_solar_generation** — Get solar production data by date range
3. **get_recent_energy_summary** — Quick 24h (or custom) usage + generation snapshot
4. **get_weather_forecast** — Weather forecast including solar irradiance estimates
5. **get_electricity_prices** — Time-of-use pricing for any date (peak/off-peak rates)
6. **search_energy_tips** — Semantic search through energy-saving best practices
7. **calculate_energy_savings** — Compute kWh/USD savings for optimization scenarios

## Constraints
- Always cite data sources when making claims (e.g., "Based on your usage last
month...")
- Never fabricate numbers — use actual tool outputs or clearly state estimates
- If critical data is missing (e.g., device wattage, tariff info), ask the user before
guessing
- Do not recommend unsafe actions (e.g., modifying electrical systems without a
professional)
- Respect user privacy — never store or share personal data beyond the current session
- If asked about topics outside energy (medical, legal, financial), politely redirect

## Response Best Practices
- Start with the most impactful recommendation, then detail the reasoning
- Include estimated savings (kWh and USD) when data supports it
- Explain tradeoffs (e.g., "This reducespeak usage but may require changing your
schedule")
- Use structured formatting: bullet points for tips, tables for comparisons, bold for
key numbers
- End with a brief summary of what the user should do next
- Always ask if the user wants to explore another area (solar, specific appliances, bill
breakdown)

## User Interaction Style
- Acknowledge the user's goals upfront before diving into analysis
- Use their home data whenever possible — personalization builds trust
- Propose concrete next steps, not just "let me know if you need help"
- If the user asks a vague question, clarify before answering (e.g., "Are you asking
about summer cooling costs or year-round usage?")

## Tone and Voice
- Helpful, encouraging, not judgmental
- Avoid jargon — explain terms like "demand charge" or "peak hours" when first used
- Be concise — respect the user's time

"""

In [67]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [68]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [69]:
print(response["messages"][-1].content)

Here's the optimal charging strategy for your electric car tomorrow (October 7, 2023) based on electricity prices and solar power generation in San Francisco:

### Electricity Prices
- **Off-Peak (Lowest Rate: $0.08/kWh)**: 
  - 12 AM - 5 AM
  - 10 PM - 11 PM
- **Part-Peak (Moderate Rate: $0.12/kWh)**: 
  - 6 AM - 9 AM
  - 2 PM - 4 PM
- **Peak (Highest Rate: $0.18/kWh)**: 
  - 10 AM - 1 PM
  - 4 PM - 9 PM

### Solar Power Generation Forecast
- **Peak Solar Generation (High Irradiance)**:
  - 11 AM - 1 PM: Solar irradiance is expected to be around **592.8 W/m²** at 11 AM and **363.8 W/m²** at 12 PM.
  - 2 PM - 4 PM: Solar irradiance will be lower but still significant, around **567.1 W/m²** at 1 PM and **325.4 W/m²** at 2 PM.

### Recommended Charging Times
1. **Charge During Off-Peak Hours**:
   - **12 AM - 5 AM**: Charge your car overnight at the lowest rate of **$0.08/kWh**.
   
2. **Charge During Part-Peak Hours**:
   - **2 PM - 4 PM**: If you need to charge during the day, consider

In [70]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_electricity_prices
- get_weather_forecast


## 2. Define Test Cases

In [71]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [72]:
test1 = {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
        }

In [73]:
test2 = {
        "id": "thermostat_1",
        "question": "What's the best thermostat setting for summer to balance comfort and energy savings?",
        "expected_tools": ["search_energy_tips", "get_weather_forecast"],
        "expected_response": "Should include specific temperature recommendation, estimated savings, and reasoning",
    }

In [74]:
test3 = {
        "id": "appliance_scheduling_1",
        "question": "When is the best time to run my dishwasher and washing machine to save money?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should recommend off-peak hours and explain time-of-use pricing",
    }

In [75]:
test4 = {
        "id": "solar_optimization_1",
        "question": "How can I maximize my solar power usage during the day?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation", "search_energy_tips"],
        "expected_response": "Should include timing recommendations for high-consumption appliances and solar production patterns",
    }

In [76]:
test5 = {
        "id": "cost_analysis_1",
        "question": "I used 500 kWh last month and my bill was $65. Is that reasonable for a 2-bedroom apartment?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices"],
        "expected_response": "Should compare against average usage, explain rate tiers, and provide context",
    }

In [77]:
test6 = {
        "id": "hvac_1",
        "question": "My AC runs constantly during hot afternoons. What can I do to reduce energy use without sacrificing comfort?",
        "expected_tools": ["get_recent_energy_summary", "get_weather_forecast", "search_energy_tips"],
        "expected_response": "Should identify HVAC as high consumer and provide specific optimization tips",
    }

In [78]:
test7 = {
        "id": "peak_vs_offpeak_1",
        "question": "What's the difference between peak and off-peak electricity rates, and why does it matter for my bill?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should clearly explain time-of-use tiers, hours, and provide cost impact examples",
    }

In [79]:
test8 = {
        "id": "ev_charging_2",
        "question": "I have a Tesla Model 3 with 60 kWh battery. How much would it cost to fully charge at home?",
        "expected_tools": ["calculate_energy_savings", "get_electricity_prices"],
        "expected_response": "Should calculate cost based on current rates and explain variables (battery size, efficiency)",
    }

In [80]:
test9 = {
        "id": "pool_pump_1",
        "question": "I have a swimming pool with a 1.5 HP pump. What's the most energy-efficient schedule for running it?",
        "expected_tools": ["search_energy_tips", "get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "Should recommend off-peak scheduling, estimate daily/weekly cost, and discuss runtime optimization",
    }

In [81]:
test10 = {
        "id": "energy_audit_1",
        "question": "What are the biggest energy wasters in a typical home and what are the quick fixes I can do myself?",
        "expected_tools": ["search_energy_tips", "get_recent_energy_summary"],
        "expected_response": "Should list common culprits (phantom load, HVAC, lighting), provide actionable tips with estimated savings",
    }

In [82]:
test_cases = [ test1, test2, test3, test4, test5, test6, test7, test8, test9, test10 ]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [83]:
CONTEXT = "Location: San Francisco, CA"

In [84]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: thermostat_1
Question: What's the best thermostat setting for summer to balance comfort and energy savings?
--------------------------------------------------

Test 3: appliance_scheduling_1
Question: When is the best time to run my dishwasher and washing machine to save money?
--------------------------------------------------

Test 4: solar_optimization_1
Question: How can I maximize my solar power usage during the day?
--------------------------------------------------

Test 5: cost_analysis_1
Question: I used 500 kWh last month and my bill was $65. Is that reasonable for a 2-bedroom apartment?
--------------------------------------------------

Test 6: hvac_1
Question: My AC runs constantly during hot afternoons. What can I do to reduce energy use without sacrificing comfort?

In [85]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='aa303c0b-9275-4556-bd21-de46b1591e2b'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='6cc033a8-cb32-4b2a-bbce-5bd00df0bd2f'),
    AIMessage(content="To provide you with the best charging times for your electric car tomorrow, I'll need to check the electricity prices for tomorrow and the weather forecast, including solar irradiance estimates. This will help us identify the optimal charging window when solar power generation is at its peak and electricity costs are lower.\n\nLet me gather that information for you. One moment, please!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'com

## 4. Evaluate Responses

In [ ]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [86]:
def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected criteria.

    Args:
        question (str): The original question asked
        final_response (str): The agent's final response text
        expected_response (str): Description of what was expected

    Returns:
        dict: Evaluation score and feedback
    """
    score = 0
    max_score = 5
    feedback = []

    # Check 1: Response is not empty
    if final_response and len(final_response.strip()) > 0:
        score += 1
        feedback.append("✓ Response is not empty")
    else:
        feedback.append("✗ Response is empty")

    # Check 2: Response addresses the question (contains relevant keywords)
    question_lower = question.lower()
    response_lower = final_response.lower()

    # Key topics to look for
    key_topics = {
        "ev_charging": ["charge", "charging", "electric", "vehicle", "battery", "tesla"],
        "thermostat": ["thermostat", "temperature", "cooling", "heating", "ac", "hvac"],
        "appliance": ["dishwasher", "washing", "machine", "appliance", "run"],
        "solar": ["solar", "generation", "panels", "irradiance"],
        "cost": ["cost", "dollar", "price", "rate", "bill", "kwh"],
        "peak": ["peak", "off-peak", "offpeak", "rate", "tier"],
        "hvac": ["hvac", "ac", "air", "conditioning", "cooling", "heating"],
        "pool": ["pool", "pump", "swimming"],
        "audit": ["waster", "phantom", "standby", "bulb", "led"],
    }

    # Find relevant topics based on question
    relevant_topics = []
    for topic, keywords in key_topics.items():
        if any(kw in question_lower for kw in keywords):
            relevant_topics.append(topic)

    # Check if response covers relevant topics
    topics_covered = sum(1 for t in relevant_topics if any(kw in response_lower for kw in key_topics[t]))

    if topics_covered >= len(relevant_topics) * 0.5:  # At least 50% covered
        score += 1
        feedback.append(f"✓ Covers relevant topics ({topics_covered}/{len(relevant_topics)})")
    else:
        feedback.append(f"✗ Missing relevant topics ({topics_covered}/{len(relevant_topics)})")

    # Check 3: Contains actionable advice (numbers, recommendations)
    has_numbers = any(c.isdigit() for c in final_response)
    has_recommendation = any(word in response_lower for word in ["recommend", "suggest", "should", "best", "optimal", "try", "consider"])

    if has_numbers:
        score += 1
        feedback.append("✓ Contains specific numbers/data")
    else:
        feedback.append("✗ Missing specific numbers/data")

    if has_recommendation:
        score += 1
        feedback.append("✓ Contains actionable recommendations")
    else:
        feedback.append("✗ Missing actionable recommendations")

    # Check 4: Explains reasoning (has explanation words)
    has_reasoning = any(word in response_lower for word in ["because", "since", "therefore", "this means", "reason", "explain"])
    if has_reasoning:
        score += 1
        feedback.append("✓ Explains reasoning")
    else:
        feedback.append("✗ Missing reasoning explanation")

    return {
        "score": score,
        "max_score": max_score,
        "percentage": round(score / max_score * 100, 1),
        "feedback": feedback
    }

In [87]:
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used.

    Args:
        messages (list): List of messages from agent response
        expected_tools (list): List of tool names that were expected to be called

    Returns:
        dict: Evaluation score and details
    """
    # Extract actual tool calls from messages
    actual_tools = []
    for msg in messages:
        msg_dict = msg.model_dump() if hasattr(msg, 'model_dump') else msg
        if msg_dict.get("tool_call_id"):
            # This is a tool result - get the tool name
            tool_name = msg.name if hasattr(msg, 'name') else msg_dict.get("name", "unknown")
            actual_tools.append(tool_name)

    score = 0
    max_score = len(expected_tools) if len(expected_tools) > 0 else 1
    feedback = []

    # Check if expected tools were called
    if expected_tools:
        matched = [t for t in expected_tools if t in actual_tools]
        missing = [t for t in expected_tools if t not in actual_tools]
        extra = [t for t in actual_tools if t not in expected_tools]

        score = len(matched)
        max_score = len(expected_tools)

        if matched:
            feedback.append(f"✓ Correct tools used: {matched}")
        if missing:
            feedback.append(f"✗ Missing tools: {missing}")
        if extra:
            feedback.append(f"ℹ Extra tools used: {extra}")

        percentage = round(score / max_score * 100, 1) if max_score > 0 else 0
    else:
        percentage = 100 if not actual_tools else 50
        feedback.append("No specific tools expected")

    return {
        "score": score,
        "max_score": max_score,
        "percentage": percentage,
        "expected_tools": expected_tools,
        "actual_tools": actual_tools,
        "feedback": feedback
    }

In [88]:
def generate_evaluation_report(test_results):
    """Generate a comprehensive evaluation report.

    Args:
        test_results (list): List of test results from agent runs

    Returns:
        dict: Comprehensive evaluation report with metrics and recommendations
    """
    report = {
        "total_tests": len(test_results),
        "response_evaluation": {
            "total_score": 0,
            "total_max": 0,
            "average_percentage": 0,
            "details": []
        },
        "tool_usage_evaluation": {
            "total_score": 0,
            "total_max": 0,
            "average_percentage": 0,
            "details": []
        },
        "strengths": [],
        "weaknesses": [],
        "recommendations": []
    }

    response_scores = []
    tool_scores = []

    for result in test_results:
        question = result.get("question", "")
        expected_response = result.get("expected_response", "")
        expected_tools = result.get("expected_tools", [])

        # Get response messages
        if "error" in str(result.get("response", "")).lower():
            # Error case - skip evaluation
            continue

        response_obj = result.get("response", {})
        messages = response_obj.get("messages", [])

        # Get final response (last AI message)
        final_response = ""
        for msg in reversed(messages):
            if hasattr(msg, "type") and msg.type == "ai":
                final_response = msg.content
                break
            elif hasattr(msg, "content") and "content" in msg.additional_kwargs:
                # Check if it's an AI message
                if not hasattr(msg, "tool_call_id"):
                    final_response = msg.content
                    break

        # Evaluate response quality
        resp_eval = evaluate_response(question, final_response, expected_response)
        report["response_evaluation"]["details"].append({
            "test_id": result["test_id"],
            **resp_eval
        })
        response_scores.append(resp_eval["percentage"])

        # Evaluate tool usage
        tool_eval = evaluate_tool_usage(messages, expected_tools)
        report["tool_usage_evaluation"]["details"].append({
            "test_id": result["test_id"],
            **tool_eval
        })
        tool_scores.append(tool_eval["percentage"])

    # Calculate averages
    if response_scores:
        report["response_evaluation"]["average_percentage"] = round(sum(response_scores) / len(response_scores), 1)
    if tool_scores:
        report["tool_usage_evaluation"]["average_percentage"] = round(sum(tool_scores) / len(tool_scores), 1)

    # Identify strengths
    if report["response_evaluation"]["average_percentage"] >= 70:
        report["strengths"].append("Good response quality overall")
    if report["tool_usage_evaluation"]["average_percentage"] >= 70:
        report["strengths"].append("Appropriate tool usage")

    # Identify weaknesses
    if report["response_evaluation"]["average_percentage"] < 70:
        report["weaknesses"].append("Response quality needs improvement")
    if report["tool_usage_evaluation"]["average_percentage"] < 70:
        report["weaknesses"].append("Tool selection needs improvement")

    # Generate recommendations
    if report["response_evaluation"]["average_percentage"] < 50:
        report["recommendations"].append("Improve response completeness with more specific data and recommendations")
    if report["tool_usage_evaluation"]["average_percentage"] < 50:
        report["recommendations"].append("Review tool selection logic in agent instructions")

    report["recommendations"].append("Consider adding more test cases for edge cases")
    report["recommendations"].append("Implement human feedback collection for real-world validation")

    return report

In [89]:

# Generate evaluation report
evaluation_report = generate_evaluation_report(test_results)

print("=" * 60)
print("EVALUATION REPORT")
print("=" * 60)
print("Total Tests:", evaluation_report['total_tests'])
print("Response Quality:", evaluation_report['response_evaluation']['average_percentage'])
print("Tool Usage:", evaluation_report['tool_usage_evaluation']['average_percentage'])

print("\n--- Strengths ---")
for s in evaluation_report['strengths']:
    print(f"  ✓ {s}")

print("\n--- Weaknesses ---")
for w in evaluation_report['weaknesses']:
    print(f"  ✗ {w}")

print("\n--- Recommendations ---")
for r in evaluation_report['recommendations']:
    print(f"  → {r}")

EVALUATION REPORT
Total Tests: 10
Response Quality: 78.0
Tool Usage: 20.0

--- Strengths ---
  ✓ Good response quality overall

--- Weaknesses ---
  ✗ Tool selection needs improvement

--- Recommendations ---
  → Review tool selection logic in agent instructions
  → Consider adding more test cases for edge cases
  → Implement human feedback collection for real-world validation
